In [ ]:
import math
import matplotlib.pyplot as plt
import networkx as nx
import heapq

coordinates = {
    "Pharmacy":        (0, 0),
    "Patient_Wing":    (1, 4),
    "Main_Corridor":   (2, 1),
    "Nursing_Station": (4, 2),
    "Laboratory":      (5, 5),
    "Emergency_Ward":  (8, 6),
}

edges_with_costs = [
    ("Pharmacy", "Patient_Wing", 4.1),
    ("Pharmacy", "Main_Corridor", 2.2),
    ("Main_Corridor", "Nursing_Station", 2.2),
    ("Patient_Wing", "Laboratory", 5.0),
    ("Nursing_Station", "Laboratory", 3.2),
    ("Nursing_Station", "Emergency_Ward", 6.0),
    ("Laboratory", "Emergency_Ward", 3.2),
]

G = nx.DiGraph()
for node, pos in coordinates.items():
    G.add_node(node, pos=pos)
for u, v, w in edges_with_costs:
    G.add_edge(u, v, weight=w)

start = "Pharmacy"
goal  = "Emergency_Ward"

def euclidean(a, b):
    (x1, y1), (x2, y2) = coordinates[a], coordinates[b]
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

heuristics = {n: round(euclidean(n, goal), 4) for n in coordinates}

print("Heuristic h(n) to Emergency_Ward:")
print(f"{'Location':<18} | h(n)")
print("-" * 30)
for n, h in heuristics.items():
    print(f"{n:<18} | {h}")

def astar(graph, start, goal, h):
    counter = 0
    open_list = [(h[start], counter, start, [start], 0)]
    visited = set()
    expansion_order = []
    best_g = {start: 0}
    trace = []

    while open_list:
        f, _, current, path, g = heapq.heappop(open_list)

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)
        trace.append((current, g, h[current], f))

        if current == goal:
            return path, g, expansion_order, trace

        for neighbor in graph.neighbors(current):
            edge_cost = graph[current][neighbor]["weight"]
            new_g = g + edge_cost

            if neighbor not in best_g or new_g < best_g[neighbor]:
                best_g[neighbor] = new_g
                counter += 1
                new_f = new_g + h[neighbor]
                heapq.heappush(
                    open_list,
                    (new_f, counter, neighbor, path + [neighbor], new_g)
                )
    return None, float("inf"), expansion_order, trace


path, total_cost, expansion, trace = astar(G, start, goal, heuristics)

print("\n--- A* Results ---")
print("Node expansion order :", " -> ".join(expansion))
print("Solution path        :", " -> ".join(path))
print("Total path cost      :", total_cost)

print(f"\n{'Node':<18} | {'g(n)':>6} | {'h(n)':>6} | {'f(n)':>6}")
print("-" * 46)
for node, g, h, f in trace:
    print(f"{node:<18} | {g:>6} | {h:>6} | {f:>6}")

plt.figure(figsize=(11, 6))
pos = nx.get_node_attributes(G, "pos")

nx.draw_networkx_nodes(G, pos, node_size=1400,
                       node_color="lightblue", edgecolors="black")
nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=18,
                       edge_color="lightgray", width=1.8)

path_edges = list(zip(path, path[1:]))
nx.draw_networkx_edges(G, pos, edgelist=path_edges,
                       arrowstyle="->", arrowsize=22,
                       edge_color="red", width=3.0)

nx.draw_networkx_nodes(G, pos, nodelist=path, node_size=1500,
                       node_color="orange", edgecolors="black")

labels = {n: f"{n}\n{coordinates[n]}\nh={heuristics[n]}" for n in coordinates}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7)

edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9)

plt.title("Task 3: A* on Hospital Corridors\n"
          "Path: " + " -> ".join(path) +
          f"   |   Total Cost = {total_cost}",
          fontsize=11, fontweight="bold")
plt.axis("on")
plt.grid(True)
plt.tight_layout()
plt.show()